- Pyro-SDIS RF-DETR-L resume | Kaggle | 2 GPU T4
  - Mục đích: chạy tiếp 12 epoch còn lại (8/20 → 20/20) từ `checkpoint_7.ckpt` của run GB10 `rfdetr_large_pyro_sdis_gb10`
  - Kaggle Input cần: dataset `pyro-sdis-yolo` (layout `images/{train,val}`, `labels/{train,val}`) + checkpoint resume `.ckpt`; nếu resume qua session mới, đặt `metrics.csv` cạnh checkpoint để giữ history
  - Protocol 13f khóa: 1280 px, 20 epoch, seed 20260707, effective batch 16, augmentation mặc định RF-DETR (không override), không early stop
  - Sai khác bắt buộc so với GB10 (đổi hardware): devices 1→2 (micro batch 4×accum4 → 2/GPU×2GPU×accum4, effective giữ 16); amp `auto`(bf16) → `fp16` (T4 không hỗ trợ bf16); num_workers 4→2
  - Cảnh báo: args nhúng trong checkpoint GB10 cho thấy run gốc chạy `early_stopping=true` (callback `RFDETREarlyStopping` nằm trong ckpt) — notebook này ép `early_stopping=False` theo policy đủ 20 epoch; Lightning sẽ warning bỏ qua state callback đó, chấp nhận được
  - Giữ nguyên GB10: constructor `RFDETRLarge(resolution=1280)` không truyền num_classes (tự suy 1 class từ data.yaml); không override lr_scheduler/warmup (mặc định `step`, lr_drop 100 → LR không drop trong 20 epoch)
  - Shim dataset: notebook tự tạo symlink layout `<split>/<images|labels>` + `data.yaml` (nc:1 smoke) giống `ensure_shim_dataset()` của `train_rfdetr.py` cũ
  - Resume: tự quét working + Kaggle Input, chọn checkpoint full-state epoch cao nhất; đủ 20/20 thì không train lại; nếu OOM → hạ MICRO_BATCH=1, GRAD_ACCUM=8
  - Output: `/kaggle/working/runs/rfdetr_large_pyro_sdis`


In [1]:
!find /kaggle/input -maxdepth 10 -type d | head -100


/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/registerusername
/kaggle/input/datasets/registerusername/checkpoint
/kaggle/input/datasets/registerusername/pyro-sdis-yolo
/kaggle/input/datasets/registerusername/pyro-sdis-yolo/datasets
/kaggle/input/datasets/registerusername/pyro-sdis-yolo/datasets/smoke_fire_detection
/kaggle/input/datasets/registerusername/pyro-sdis-yolo/datasets/smoke_fire_detection/pyro-sdis-yolo
/kaggle/input/datasets/registerusername/pyro-sdis-yolo/datasets/smoke_fire_detection/pyro-sdis-yolo/labels
/kaggle/input/datasets/registerusername/pyro-sdis-yolo/datasets/smoke_fire_detection/pyro-sdis-yolo/labels/val
/kaggle/input/datasets/registerusername/pyro-sdis-yolo/datasets/smoke_fire_detection/pyro-sdis-yolo/labels/train
/kaggle/input/datasets/registerusername/pyro-sdis-yolo/datasets/smoke_fire_detection/pyro-sdis-yolo/images
/kaggle/input/datasets/registerusername/pyro-sdis-yolo/datasets/smoke_fire_detection/pyro-sdis-yolo/images/val
/kaggle/input/datas

In [2]:
from pathlib import Path
import shutil

SEED = 20260707
EPOCHS = 20
RESOLUTION = 1280
MICRO_BATCH = 2
GRAD_ACCUM = 4
INPUT_BASE = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
RUN_NAME = 'rfdetr_large_pyro_sdis'
RUN_DIR = WORK_ROOT / 'runs' / RUN_NAME
SHIM_DIR = WORK_ROOT / 'pyro-sdis-yolo-rfdetr'

# Xoá thư mục rác do chạy lỗi trước đó
shutil.rmtree(SHIM_DIR, ignore_errors=True)

dataset_candidates = sorted({path.parent.resolve() for path in INPUT_BASE.rglob('images')
                             if (path / 'train').is_dir() and (path / 'val').is_dir()
                             and (path.parent / 'labels' / 'train').is_dir()
                             and (path.parent / 'labels' / 'val').is_dir()})
assert len(dataset_candidates) == 1, f'Cần đúng 1 dataset, thấy: {dataset_candidates}'
DATA_SRC = dataset_candidates[0]

# Ánh xạ đúng tên thư mục và biến src
for src, dst_name in (('train', 'train'), ('val', 'valid')):
    for sub in ('images', 'labels'):
        dst = SHIM_DIR / dst_name / sub
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.is_symlink():
            dst.symlink_to(DATA_SRC / sub / src, target_is_directory=True)

(SHIM_DIR / 'data.yaml').write_text('nc: 1\nnames:\n  0: smoke\n')

train_count = sum(1 for _ in (SHIM_DIR / 'train' / 'images').iterdir())
# Đếm trong thư mục 'valid' thay vì 'val'
val_count = sum(1 for _ in (SHIM_DIR / 'valid' / 'images').iterdir())

assert (train_count, val_count) == (29537, 4099), f'Lệch audit: train={train_count}, val={val_count}'
print({'data_src': str(DATA_SRC), 'shim': str(SHIM_DIR), 'run_dir': str(RUN_DIR), 'train': train_count, 'val': val_count})


{'data_src': '/kaggle/input/datasets/registerusername/pyro-sdis-yolo/datasets/smoke_fire_detection/pyro-sdis-yolo', 'shim': '/kaggle/working/pyro-sdis-yolo-rfdetr', 'run_dir': '/kaggle/working/runs/rfdetr_large_pyro_sdis', 'train': 29537, 'val': 4099}


In [3]:
import subprocess
import sys

subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rfdetr[train]==1.8.3'], check=True)

import importlib.metadata
import torch

devices = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'rfdetr': importlib.metadata.version('rfdetr'), 'devices': devices})
assert len(devices) == 2 and all('T4' in name for name in devices), f'Cần Kaggle 2 GPU T4, hiện có {devices}'


Fri Jul 17 05:03:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from collections.abc import Mapping
import json
import os
import shutil

PROTOCOL = {'schema': 1, 'framework': 'rfdetr-1.8.3', 'model': 'RFDETRLarge', 'dataset': 'pyro-sdis-yolo', 'resolution': RESOLUTION, 'epochs': EPOCHS, 'seed': SEED, 'micro_batch': MICRO_BATCH, 'grad_accum': GRAD_ACCUM, 'devices': 2, 'strategy': 'ddp_notebook', 'amp_dtype': 'fp16', 'augmentation_policy': 'framework-default-no-user-overrides', 'source_run': 'rfdetr_large_pyro_sdis_gb10-epoch0-7-devices1-batch4-accum4-amp-auto'}
protocol = RUN_DIR / 'resume_protocol.json'
existed = RUN_DIR.exists()

def valid_checkpoint(path, source):
    state = torch.load(path, map_location='cpu', weights_only=False)
    if not isinstance(state, Mapping) or not isinstance(state.get('epoch'), int) or not isinstance(state.get('global_step'), int):
        raise ValueError('thiếu epoch/global_step')
    if not isinstance(state.get('state_dict'), Mapping) or not state['state_dict'] or not state.get('optimizer_states') or not state.get('lr_schedulers') or not isinstance(state.get('loops', {}).get('fit_loop'), Mapping):
        raise ValueError('không full-state')
    if not 7 <= state['epoch'] < EPOCHS:
        raise ValueError(f'epoch ngoài khoảng resume hợp lệ: {state["epoch"]}')
    if source == 'working' and (not protocol.is_file() or json.loads(protocol.read_text(encoding='utf-8')) != PROTOCOL):
        raise ValueError('protocol working không khớp')
    return {'path': path, 'source': source, 'epoch': state['epoch'], 'step': state['global_step']}

def scan_checkpoints(paths, source):
    accepted = []
    for path in paths:
        try:
            accepted.append(valid_checkpoint(path, source))
        except Exception as error:
            print('reject', source, path, error)
    return accepted

if existed and not protocol.is_file():
    raise RuntimeError('working thiếu protocol')

working_checkpoints = scan_checkpoints(RUN_DIR.glob('checkpoint_*.ckpt'), 'working')
input_checkpoints = scan_checkpoints(INPUT_BASE.rglob('checkpoint_*.ckpt'), 'input')

chosen = max(working_checkpoints + input_checkpoints, key=lambda item: (item['epoch'], item['step'], item['source'] == 'working'), default=None)
assert chosen, 'Notebook này chỉ để resume — cần checkpoint_7.ckpt (hoặc mới hơn) trong Kaggle Input hoặc working'

if chosen['source'] == 'input':
    destination = RUN_DIR / 'input_epoch_{:03d}_step_{:09d}.ckpt'.format(chosen['epoch'], chosen['step'])
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix('.tmp')
    shutil.copyfile(chosen['path'], temporary)
    os.replace(temporary, destination)
    resume_path = destination
else:
    resume_path = chosen['path']

history_path = RUN_DIR / f"metrics_through_epoch_{chosen['epoch']:02d}.csv"
metrics_source = RUN_DIR / 'metrics.csv'
if not metrics_source.is_file() and chosen['source'] == 'input':
    metrics_source = chosen['path'].with_name('metrics.csv')
if metrics_source.is_file() and not history_path.is_file():
    shutil.copyfile(metrics_source, history_path)

if not protocol.is_file():
    protocol.write_text(json.dumps(PROTOCOL, indent=2, sort_keys=True), encoding='utf-8')

run_complete = chosen['epoch'] >= EPOCHS - 1
print({'resume_source': chosen['source'], 'resume_path': str(resume_path), 'epoch': chosen['epoch'], 'complete': run_complete, 'metrics_history': str(history_path) if history_path.is_file() else None})


{'resume_source': 'input', 'resume_path': '/kaggle/working/runs/rfdetr_large_pyro_sdis/input_epoch_007_step_000014776.ckpt', 'epoch': 7, 'complete': False}


In [5]:
from rfdetr import RFDETRLarge

if run_complete:
    print(f'Không train lại: checkpoint đã hoàn thành epoch {EPOCHS}')
else:
    model = RFDETRLarge(resolution=RESOLUTION)
    model.train(dataset_dir=str(SHIM_DIR), dataset_file='yolo', output_dir=str(RUN_DIR), epochs=EPOCHS, batch_size=MICRO_BATCH, grad_accum_steps=GRAD_ACCUM, accelerator='gpu', devices=2, strategy='ddp_notebook', amp_dtype='fp16', num_workers=2, checkpoint_interval=1, seed=SEED, early_stopping=False, tensorboard=False, resume=str(resume_path))


[2026-07-17 05:04:19] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-large-2026.pth


/usr/local/lib/python3.12/dist-packages/rfdetr/models/weights.py:258: FutureWarning: target=True is deprecated since `v0.8`; use `TargetMode.ARGS_REMAP` instead. Will be removed in `v1.0`.
  @deprecated(target=True, args_mapping={"train_config": None}, deprecated_in="1.7.0", remove_in="1.9.0", num_warns=-1)


/root/.roboflow/models/rf-detr-large-2026.pth:   0%|          | 0.00/130M [00:00<?, ?iB/s]

[2026-07-17 05:04:20] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-large-2026.pth


[2026-07-17 05:04:20] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-17 05:04:20] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


...

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:433: It is recommended to use `self.log('train/loss_ce', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:433: It is recommended to use `self.log('train/class_error', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:433: It is recommended to use `self.log('train/loss_bbox', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:433: It is recommended to use `self.log('train/loss_giou', ..., s

              Val (Epoch 10/20) — Overall Metrics               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.4298 │ 0.7291 │ 0.4521 │ 0.7142 │ 0.7224 │ 0.7052 │ 0.7405 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘
           Val (Epoch 10/20) — Per-class Metrics           
┏━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Class ┃ AP 50:95 ┃     AR ┃     F1 ┃ Precision ┃ Recall ┃
┡━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ smoke │   0.4298 │ 0.7142 │ 0.7224 │    0.7052 │ 0.7405 │
└───────┴──────────┴────────┴────────┴───────────┴────────┘



Detected KeyboardInterrupt, attempting graceful shutdown ...
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/call.py", line 48, in _call_and_handle_interrupt
    return trainer.strategy.launcher.launch(trainer_fn, *args, trainer=trainer, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/strategies/launchers/multiprocessing.py", line 144, in launch
    while not process_context.join():
              ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/spawn.py", line 140, in join
    ready = multiprocessing.connection.wait(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self._selector.poll(timeout)
     

TypeError: object of type 'NoneType' has no len()

In [ ]:
checkpoints = sorted(RUN_DIR.glob('checkpoint_*.ckpt'))
weights = sorted(RUN_DIR.glob('*.pth'))
csvs = sorted(RUN_DIR.glob('metrics*.csv'))
for path in checkpoints + weights + csvs:
    print(path, path.stat().st_size)
assert checkpoints
assert weights
